# 古い対話を要約して長い対話を続ける

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/19_dialogue/chat_summary_memory.ipynb)

NLP クックブック 19-06 ／ 対話システム

長い対話で、古いターンを単に捨てると大事な情報を忘れてしまう。捨てる前に要約して、記憶として持ち越したい。

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(name)
llm = AutoModelForCausalLM.from_pretrained(name).eval()

def generate(messages, max_new_tokens=100):
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

class SummaryChat:
    def __init__(self, system, keep_turns=2):
        self.system, self.summary, self.history, self.keep = system, "", [], keep_turns * 2

    def compress(self):
        """直近 keep_turns ターンより古い部分を、これまでの要約と合わせて要約し直す"""
        old, self.history = self.history[:-self.keep], self.history[-self.keep:]
        dialogue = "\n".join(f"{'ユーザー' if m['role'] == 'user' else 'アシスタント'}: {m['content']}" for m in old)
        prompt = ("次の「これまでの要約」と「対話」から、ユーザーの名前・予定・好み・決まったことを漏らさずに、3 文以内の新しい要約を書いてください。\n\n"
                  f"# これまでの要約\n{self.summary or '(なし)'}\n\n# 対話\n{dialogue}")
        self.summary = generate([{"role": "user", "content": prompt}], max_new_tokens=120)

    def say(self, text):
        self.history.append({"role": "user", "content": text})
        if len(self.history) > self.keep + 2:
            self.compress()
        system = self.system + (f"\n\nこれまでの対話の要約: {self.summary}" if self.summary else "")
        reply = generate([{"role": "system", "content": system}] + self.history, max_new_tokens=80)
        self.history.append({"role": "assistant", "content": reply})
        return reply

chat = SummaryChat("あなたは旅行の相談に乗るアシスタントです。2 文以内で簡潔に答えてください。")
for user in ["私の名前は田中です。来月、京都に 2 泊で行く予定です。", "混雑が苦手なので、静かな場所が好きです。", "秋におすすめのお寺を 1 つ教えてください。",
             "そこへは京都駅からどう行けばいいですか？", "近くでお昼を食べるなら何がいいですか？", "ところで、私の名前と、どんな場所が好きかを覚えていますか？"]:
    print(f"あなた: {user}\nボット: {chat.say(user)}")
    if chat.summary:
        print(f"   [要約] {chat.summary}")

あなた: 私の名前は田中です。来月、京都に 2 泊で行く予定です。
ボット: 田中様、京都へのご旅行楽しみですね！来月の京都では、清水寺や嵐山、金閣寺などの観光スポットを訪れるのはいかがでしょうか。また、京都の食文化もお楽しみください。例えば、湯豆腐や抹茶スイーツなどがおすすめです。素敵な旅になりますように！
あなた: 混雑が苦手なので、静かな場所が好きです。
ボット: 田中様、京都には混雑を避けられる静かな場所もたくさんあります。例えば、哲学の道や真如堂、詩仙堂などはいかがでしょうか。これらの場所は自然豊かで、落ち着いた雰囲気を楽しむことができます。ぜひご検討ください。
あなた: 秋におすすめのお寺を 1 つ教えてください。
ボット: 田中様、秋に京都でおすすめなのは嵯峨野の「祇王寺」です。紅葉の時期、苔むした庭に色鮮やかなもみじが映え、とても幻想的な雰囲気を楽しむことができます。ぜひ訪れてみてください。
あなた: そこへは京都駅からどう行けばいいですか？
ボット: 祇王寺は京都駅から少し離れているので、バスを利用するのが便利です。京都駅前バスターミナルから市バス「71」または「84」系統に乗車し、「嵯峨釈迦堂前」で下車してください。そこから徒歩約15分で祇王寺に到着します。

または、京都市営地下鉄東西線の「太秦天神川駅」で下車し、そこから市バス
   [要約] 田中様は来月京都に2泊する予定で、清水寺や嵐山、金閣寺といった観光スポットを訪れる計画を立てています。特に混雑を避けるため、静かな場所を好むことから、湯豆腐や抹茶スイーツといった京都の食文化も楽しむ予定です。この旅では、田中様が希望する静かで落ち着いた雰囲気の中で、京都の魅力を存分に味わうことができるでしょう。
あなた: 近くでお昼を食べるなら何がいいですか？
ボット: 祇王寺の周辺でお昼を食べるなら、嵯峨野エリアの伝統的なお店がおすすめです。特に、湯豆腐や京野菜を使った料理が楽しめるお店がいくつかあります。例えば、「清遊館」や「嵯峨野」などは人気のお店です。また、祇王寺から少し歩いたところにある「嵐山」エリアにも多くの飲食店がありますので、そちらもおすすめです。

具体的には、祇王寺
   [要約] 田中様は来月京都に2泊する予定で、清水寺や嵐山、金閣寺といった観光スポットを訪れる計画を立てています。

- 直近の数ターンはそのまま残し、それより古い部分だけを要約に置き換える。最近のやりとりは細部まで、昔のことは要点だけ覚えている状態になる。
- 要約は LLM の呼び出しが 1 回増える。毎ターンではなく、履歴が一定の長さを超えたときだけ行う。
- 要約を重ねるうちに細部は落ちていく。確実に覚えておきたい事実 (名前、日程、決定事項) は、要約とは別に項目として抽出して保存する方法もある。
- 過去の対話を検索できる形で保存しておき、関係する部分だけを取り出して入力に加える方法 (RAG と同じ考え方) もよく使われる。